In [ ]:
# ¿Qué segmentos, regiones y categorías concentran las ventas y la utilidad de Superstore?

from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
DATA = ROOT / "data"
DETAIL = ROOT / "submission" / "superstore_enriched_sales.csv"
ANSWER = ROOT / "submission" / "sales_by_segment_region_category.csv"

In [ ]:
# Las claves contextuales preservan las combinaciones de atributos que Order ID no identifica por sí solo.

orders = pd.read_csv(DATA / "orders.csv")
customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
order_lines = pd.read_csv(DATA / "order_lines.csv")

orders.shape, customers.shape, products.shape, order_lines.shape

In [ ]:
# Cada integración conserva una línea de orden y añade el contexto necesario para analizarla.

enriched_sales = (
    order_lines.merge(orders, on="order_context_key", validate="many_to_one")
    .merge(customers, on="customer_context_key", validate="many_to_one")
    .merge(products, on="product_context_key", validate="many_to_one")
)

enriched_sales.shape

In [ ]:
# La respuesta se agrega después de integrar las fuentes, no antes.

sales_by_context = (
    enriched_sales.groupby(
        ["Customer Segment", "Region", "Product Category"], as_index=False
    )
    .agg(sales=("Sales", "sum"), profit=("Profit", "sum"), order_lines=("Sales", "size"))
    .sort_values(["sales", "profit"], ascending=False)
)

sales_by_context.head()

In [ ]:
# La entrega conserva tanto el detalle integrado como la respuesta agregada para auditoría posterior.

assert len(enriched_sales) == len(order_lines)
assert enriched_sales[["Sales", "Profit", "Customer Segment", "Region", "Product Category"]].notna().all().all()

enriched_sales.to_csv(DETAIL, index=False)
sales_by_context.to_csv(ANSWER, index=False)